In [1]:
import torch
import torchvision
from torch import nn, optim
from torch.utils.data import DataLoader
from matplotlib import pyplot as plt
from torchvision import transforms, datasets
from cnn_model2 import PrecisionBalancedCNN
from ghost_simple import SimpleGhost
from MobileNetV2 import MobileNetV2
from MobileNetV3 import MobileNetV3
from ShuffleNetV2 import ShuffleNetV2
from ResNet18 import resnet18
from GhostNet_3 import GhostNet

In [2]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # 如果使用多GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed = 512
# 在训练代码开头调用
set_seed(seed) # 42, 2025, 1024, 512, 100

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307, ), (0.3081, )),
    transforms.Grayscale(1)
])

In [5]:
test_db = datasets.ImageFolder(root= 'C:/Users/Administrator/Desktop/datas/test_data_black', transform= transform)
test_loader = DataLoader(test_db, batch_size= 10, shuffle= True)
len(test_loader)

297

In [6]:
model_basic = MobileNetV3(version= 'small', in_ch= 1)
model_basic.load_state_dict(torch.load(f'models/complete/mobilenetv3_md_{seed}.pth'))
model_basic.to(device)
model_basic.eval()
print('mobilenetv3 load')

mobilenetv3 load


In [7]:
model_ghost = GhostNet(in_ch= 1)
model_ghost.load_state_dict(torch.load(f'models/complete/ghostnet_change_2_md_{seed}.pth'))
model_ghost.to(device)
model_ghost.eval()
print('ghostnet load')

ghostnet load


In [8]:
model_inverted_residual = MobileNetV2(in_ch= 1,)
model_inverted_residual.load_state_dict(torch.load(f'models/complete/mobilenetv2_md_{seed}.pth'))
model_inverted_residual.to(device)
model_inverted_residual.eval()
print('mobilenetv2 load')

mobilenetv2 load


In [9]:
model_residual = resnet18(in_ch= 1)
model_residual.load_state_dict(torch.load(f'models/complete/resnet18_md_{seed}.pth'))
model_residual.to(device)
model_residual.eval()
print('resnet18 load')

resnet18 load


In [10]:
model_shuffle = ShuffleNetV2(in_ch= 1)
model_shuffle.load_state_dict(torch.load(f'models/complete/shufflenetv2_md_{seed}.pth'))
model_shuffle.to(device)
model_shuffle.eval()
print('shufflenetv2 load')

model size is 1.0x
shufflenetv2 load


In [11]:
total = 0
idxs = []
correct1, correct2, correct3, correct4, correct5= 0, 0, 0, 0, 0
correct1s, correct2s, correct3s, correct4s, correct5s = [], [], [], [], []
with torch.no_grad():
    for idx, (x, y) in enumerate(test_loader):
        x, y = x.to(device), y.to(device)
        out1 = torch.argmax(model_basic(x), dim= -1)
        out2 = torch.argmax(model_ghost(x), dim= -1)
        out3 = torch.argmax(model_inverted_residual(x), dim= -1)
        out4 = torch.argmax(model_residual(x), dim= -1)
        out5 = torch.argmax(model_shuffle(x), dim= -1)

        total += y.size(0)
        correct1 += (out1 == y).sum().item()
        correct2 += (out2 == y).sum().item()
        correct3 += (out3 == y).sum().item()
        correct4 += (out4 == y).sum().item()
        correct5 += (out5 == y).sum().item()

        if idx % 50 == 0 or (idx + 1) == len(test_loader):
            acc1 = correct1 / total
            acc2 = correct2 / total
            acc3 = correct3 / total
            acc4 = correct4 / total
            acc5 = correct5 / total

            idxs.append(idx)
            correct1s.append(acc1), correct2s.append(acc2), correct3s.append(acc3)
            correct4s.append(acc4), correct5s.append(acc5)

            print(f'idx: {idx}, mobv3:{acc1:.4f}, ghost:{acc2:.4f}, mobv2:{acc3:.4f}, resnet18:{acc4:.4f}, shuffle:{acc5:.4f}')

idx: 0, mobv3:1.0000, ghost:0.8000, mobv2:1.0000, resnet18:0.8000, shuffle:0.7000
idx: 50, mobv3:0.9392, ghost:0.8980, mobv2:0.9608, resnet18:0.9235, shuffle:0.9176
idx: 100, mobv3:0.9485, ghost:0.8931, mobv2:0.9564, resnet18:0.9257, shuffle:0.9198
idx: 150, mobv3:0.9483, ghost:0.8854, mobv2:0.9570, resnet18:0.9166, shuffle:0.9185
idx: 200, mobv3:0.9517, ghost:0.8826, mobv2:0.9547, resnet18:0.9134, shuffle:0.9209
idx: 250, mobv3:0.9506, ghost:0.8781, mobv2:0.9530, resnet18:0.9163, shuffle:0.9179
idx: 296, mobv3:0.9512, ghost:0.8798, mobv2:0.9545, resnet18:0.9199, shuffle:0.9236


In [12]:
# with open('data_csv/result_complete.csv', 'a+') as f:
#     for idx, c1, c2, c3, c4, c5 in zip(idxs, correct1s, correct2s, correct3s, correct4s, correct5s):
#         f.writelines(f'{idx}, {seed}, {c1}, {c2}, {c3}, {c4}, {c5}\n')